In [ ]:
# You only need to run this block once per session to install Gurobi (and other libraries)

%pip install pyomo gurobipy pandas

In [ ]:
import gurobipy as gp
from gurobipy import GRB, Model

import pandas as pd
from typing import Any
import json

residents = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
]

# shift_types = [
#     "The one shift on this day",
#     # "3pm",
# ]

# dates = [
#     "2026-01-01",
#     "2026-01-02",
#     "2026-01-03",
# ]
dates = pd.date_range(start="2026-01-01", end="2026-02-01").strftime("%Y-%m-%d").tolist()
# print(dates)

num_residents: int = len(residents)
# num_shift_types: int = len(shift_types)
num_dates: int = len(dates)


# init
m: Model = Model()
# m.setParam('LogToConsole', 0)


x_rd = m.addVars(num_residents, num_dates, 
                    lb=0, ub=1, 
                    vtype=GRB.BINARY)

# set variable names
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates):
        x_rd[r, d].VarName = f"{resident}_{date}"

# one res for each date
for (d, date) in enumerate(dates):
    m.addConstr(gp.quicksum(x_rd[r, d] for (r, resident) in enumerate(residents)) == 1,
                name=f"single_shift_{date}")

# resident coverage ub of 7 across the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(gp.quicksum(x_rd[r, d] for (d, date) in enumerate(dates)) <= 7,
                name=f"coverage_ub_{resident}")
        
# obj fxn: minimize number of assignments to resident 0 (i.e., the first resident, "A")
m.setObjective(
    gp.quicksum(x_rd[r, d] for r, d in x_rd if r == 0),
    GRB.MINIMIZE
)

# solve
m.optimize()

# report
scheduleReport: dict[str, Any] = {}
for (d, date) in enumerate(dates):
    workingResidentNames = [resident for (r, resident) in enumerate(residents) if x_rd[r, d].X == 1]

    daySchedule: dict[str, list[str]] = {}
    daySchedule["hardcoded single shift"] = workingResidentNames

    scheduleReport[date] = daySchedule
    print(f"{date}:\n")
    print(f"\t{daySchedule}\n")


# print vars
# for v in m.getVars():
#     print(f"{v.VarName} = {v.X}")

# infeasible: bool = m.Status == GRB.INFEASIBLE
# print(m.Status)



========== SOLVING INSTANCE OF PER ==========
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 39 rows, 224 columns and 448 nonzeros (Min)
Model fingerprint: 0x0d37e0bc
Model has 0 linear objective coefficients
Variable types: 0 continuous, 224 integer (224 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]

Found heuristic solution: objective 0.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 1: 0 

Optimal solution found (tolerance 1.00e-04)
Best objective 0.000000000000e+00, best bound 0.000000000000e+00, gap 0.0000%
2026-01-01:

	{'hardcoded single shift': ['A']}

202